# 队列整理、时间匹配与终点缺口

首次记录的SRS作为暂定索引治疗。基线窗口为SRS前30天至当天；当天影像尚需核实检查与治疗先后顺序。
多病灶患者不被拆成独立患者。任何后续验证必须按患者分组。


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/rn_public').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from rn_public.bundle import restore_sources
print('Source restoration:', restore_sources(ROOT))
print('Project root:', ROOT.name)


Source restoration: {'restored_files': 0, 'status': 'all_source_checksums_verified'}
Project root: rn-mri-clinical-fusion


In [2]:
from rn_public.curate import curate
summary = curate(ROOT)
print(pd.Series(summary).to_string())


patients                                                               75
clinical_lesion_rows                                                  160
rn_time_records                                                        32
rn_time_patients                                                       27
unknown_rn_rows                                                       128
image_patient_dates                                                   635
image_series                                                         3111
baseline_directory_candidates                                          90
baseline_candidate_patients                                            59
baseline_candidate_rn_times                                            26
selected_patient_dates                                                107
selected_dicom_files                                                40801
selected_compressed_bytes                                      5997707470
final_eligible_lesions                

In [3]:
lesions = pd.read_csv(ROOT / 'data/processed/lesions.csv', dtype={'patient_id':str})
assert not lesions.duplicated(['patient_id','lesion_id']).any()
assert (lesions.loc[lesions.rn_time_day.isna(), 'rn_status'] == 'unknown').all()
print('RN source status:')
print(lesions.rn_status.value_counts().to_string())
print('Chronology conflicts:')
print(lesions.loc[lesions.rn_after_first_srs.eq(0), ['lesion_key','first_srs_start_day','rn_time_day']].to_string(index=False))


RN source status:
rn_status
unknown                128
recorded_event_time     32
Chronology conflicts:
lesion_key  first_srs_start_day  rn_time_day
  10024_L1                633.0        265.0
  40114_L3               3784.0        582.0


In [4]:
matches = pd.read_csv(ROOT / 'data/processed/baseline_matches.csv', dtype={'patient_id':str})
assert matches.baseline_to_srs_days.between(0,30).all()
print(matches[['lesion_key','baseline_date_code','baseline_to_srs_days','rn_status']].head(10).to_string(index=False))
print('Candidate lesions:', len(matches))
print('Candidate patients:', matches.patient_id.nunique())
print('Candidates with recorded RN timing:', matches.rn_time_day.notna().sum())
print('Final eligible lesions:', matches.final_eligible.sum())


lesion_key  baseline_date_code  baseline_to_srs_days           rn_status
  10005_L1            19020128                    24             unknown
  10020_L4            19000524                    21             unknown
  10020_L7            19020905                    27             unknown
  10029_L1            19000902                    17 recorded_event_time
  10029_L2            19030605                    19             unknown
  10030_L1            19010407                    28             unknown
  10031_L1            19001218                    10             unknown
  10033_L1            19000918                    12             unknown
  10033_L2            19000918                    13             unknown
  10035_L1            19000614                    27 recorded_event_time
Candidate lesions: 90
Candidate patients: 59
Candidates with recorded RN timing: 26
Final eligible lesions: 0


## 当前不能据此估计发病率或训练风险模型

128条缺少脑坏死时间的病灶记录保持为未知状态。它们不自动成为阴性对照。
论文的39个脑坏死病灶与表中32条时间记录尚未逐一对齐。
已有形态学和影像组学表可能含结局后影像，不能直接合并进治疗前预测模型。
